# Build models

* Goal - Model A:
    * Input: images.
    * Output: Action.

* Goal - Model B:
    * Input: images and language instruction.
    * Output: Action.

* Notes:
    * Data is uploaded from hdf5 file -> No need to use Tensorflow.
* Environment: robotic (env_robotic.yml)


In [11]:
import os
...

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF
from torch.optim import Adam, SGD



import pandas as pd
import json
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import numpy as np

# Data preparation

##### Load dataset
##### Discretize action space
##### Resize images
##### Create dataloaders.

In [3]:
#### load data slice as hdf5
path_data = str(Path.cwd().parent / "data")

data = []
metadata = {}
with h5py.File(f"{path_data}/combined_datasets.h5", "r") as f:
    for data_name in f.keys(): # access a single dataset.
        total_samples = 0
        data_grp = f[data_name]
        metadata[data_name] = {"actions_metadata": json.loads(data_grp.attrs["actions_metadata"])}
        print(f"n_episodes: {len(data_grp.keys())}")
        for episode_idx, episode_key in enumerate(data_grp.keys()): # access a single episode.
            grp = data_grp[episode_key]
            grp_action = grp["actions"]
            grp_images = grp["images"]
            
            # load data once per episode
            action_arrays = {key: grp_action[key][:] for key in grp_action.keys()}
            image_arrays  = {key: grp_images[key][:] for key in grp_images.keys()}

            # n_steps from any action array (they should all share dim 0)
            n_steps = next(iter(action_arrays.values())).shape[0]
            total_samples += n_steps
            # print(n_steps)

            # flatten episode into per-timestep dicts
            for t in range(n_steps):
                sample = {key: arr[t] for key, arr in action_arrays.items()}
                sample.update({key: arr[t] for key, arr in image_arrays.items()})
                sample["episode_idx"] = episode_idx
                sample["dataset_name"] = data_name  # useful if you later mix datasets

                data.append(sample)
                
        metadata[data_name]["image_keys"] = list(grp_images.keys())
        metadata[data_name]["action_keys"] = list(grp_action.keys())
        metadata[data_name]["total_samples"] = total_samples

'''
Each sample represents a time step.
Number of samples: sum(n_steps_episode)_episode, sum over all steps across all episodes both datasets together.
'''

n_episodes: 16
n_episodes: 16


'\nEach sample represents a time step.\nNumber of samples: sum(n_steps_episode)_episode, sum over all steps across all episodes both datasets together.\n'

In [4]:
print(metadata.keys())
print(metadata["stanford_hydra_dataset_converted_externally_to_rlds"]["total_samples"])
print(data[9760].keys())
print(data[9761].keys())
# np.stack([sample["dataset_name"] for sample in data])[9761:]

dict_keys(['stanford_hydra_dataset_converted_externally_to_rlds', 'taco_play'])
9761
dict_keys(['actions', 'image', 'episode_idx', 'dataset_name'])
dict_keys(['actions', 'rel_actions_gripper', 'rel_actions_world', 'terminate_episode', 'rgb_gripper', 'rgb_static', 'episode_idx', 'dataset_name'])


#### Discretize action space in 256 bins

Compute 1st/99th percentiles

In [5]:
N_BINS = 256

action_percentiles = {} # keep track of action percentiles per dataset per action dimension.
data_discretized = []

def discretize(values: np.ndarray, p1: np.ndarray, p99: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """
    Maps continuous values (n_samples, action_dim) to discrete bins [0, n_bins-1],
    independently per dimension, using p1/p99 as the [lower, upper] range.
    Values outside [p1, p99] are clipped to the nearest bin.
    """
    action_dim = values.shape[-1] if values.ndim > 1 else 1 # values.shape = (n_steps*n_episodes, action_dimensions).
    values = values.reshape(-1, action_dim) # reshape in case shape is (n_samples,).

    discretized = np.zeros_like(values, dtype=np.int64)

    for d in range(action_dim):
        bin_edges = np.linspace(p1[d], p99[d], n_bins + 1)
        binned = np.digitize(values[:, d], bin_edges[1:-1], right=True)
        discretized[:, d] = np.clip(binned, 0, n_bins - 1)

    return discretized.squeeze()


for data_name in metadata.keys():
    action_keys = metadata[data_name]["action_keys"]

    # filter samples belonging to this dataset only
    batch = [sample for sample in data if sample["dataset_name"] == data_name]

    action_percentiles[data_name] = {}

    # start each new sample as a shallow copy of the original (keeps images, episode_idx, dataset_name, etc.)
    batch_discretized = [dict(sample) for sample in batch]

    for action_key in action_keys:
        stacked = np.stack([sample[action_key] for sample in batch])
        if stacked.ndim == 1:
            stacked = stacked.reshape(-1, 1)

        p1 = np.percentile(stacked, 1, axis=0)
        p99 = np.percentile(stacked, 99, axis=0)
        action_percentiles[data_name][action_key] = {"p1": p1, "p99": p99}

        discretized = discretize(stacked, p1, p99, N_BINS)

        # overwrite the continuous action_key with its discretized version
        for sample_disc, disc_val in zip(batch_discretized, discretized.reshape(len(batch), -1)):
            sample_disc[action_key] = disc_val.squeeze()

        print(f"{data_name}\n{action_key}: shape={stacked.shape}\n"
              f"p1={p1}\np99={p99}\ndiscretized_range=[{discretized.min()}, {discretized.max()}]\n\n")

    data_discretized.extend(batch_discretized)

stanford_hydra_dataset_converted_externally_to_rlds
actions: shape=(9761, 7)
p1=[-0.01895591 -0.02405627 -0.02057381 -0.12112904 -0.16186282 -0.16376511
  0.        ]
p99=[0.02197458 0.02150433 0.02401781 0.12091041 0.11146203 0.16157938
 1.        ]
discretized_range=[0, 255]


taco_play
actions: shape=(1056, 7)
p1=[ 0.14739171 -0.41538738  0.19777607 -3.13861867 -0.53569646 -1.53636925
 -1.        ]
p99=[0.49058009 0.62614233 0.55302485 3.13905847 0.12491795 1.73633049
 1.        ]
discretized_range=[0, 255]


taco_play
rel_actions_gripper: shape=(1056, 7)
p1=[-0.5188651  -1.04988477 -0.87823469 -0.84330282 -0.76780331 -1.98089754
 -1.        ]
p99=[0.56948566 0.95734829 0.52781998 0.88023777 0.63672144 1.759892
 1.        ]
discretized_range=[0, 255]


taco_play
rel_actions_world: shape=(1056, 7)
p1=[-0.65884569 -1.04458029 -0.49734294 -0.74591375 -0.70896812 -1.77969494
 -1.        ]
p99=[0.47416454 1.00982402 0.86957543 0.72206968 0.77220851 1.95908995
 1.        ]
discretized_ran

### TODO: Overwrite data -> do not save data_discretized.

In [6]:
# sanity checks, mirroring the original data structure
print(len(data_discretized))
print(data_discretized[9760].keys())
print(data_discretized[9761].keys())
# print(data[9760]["actions"], "->", data_discretized[9760]["actions"])
data_discretized[9761]['actions']

10817
dict_keys(['actions', 'image', 'episode_idx', 'dataset_name'])
dict_keys(['actions', 'rel_actions_gripper', 'rel_actions_world', 'terminate_episode', 'rgb_gripper', 'rgb_static', 'episode_idx', 'dataset_name'])


array([145,  91,  12,   4, 160, 135, 255])

#### Filter subset of variables from each dataset, so they have same keys 

In [7]:
def filter_keys(
    data: list[dict],
    keys_to_keep: dict[str, list[str]],
    renaming_keys: dict[str, list[tuple[str, str]]] = None,
) -> list[dict]:
    """
    Filters each sample in `data` down to a subset of keys, chosen per-dataset,
    and optionally renames some of those keys per-dataset.

    Args:
        data: list of sample dicts, each with a "dataset_name" field.
        keys_to_keep: maps dataset_name -> list of keys to preserve for
            samples belonging to that dataset (using the ORIGINAL key names).
        renaming_keys: maps dataset_name -> list of (old_name, new_name) tuples.
            Applied AFTER filtering, so old_name must be one of the kept keys.
            e.g.:
            {
                "taco_play": [("rgb_static", "image")],
            }

    Returns:
        A new list of filtered (and possibly renamed) dicts. Original `data`
        is left untouched.
    """
    renaming_keys = renaming_keys or {}
    filtered = []

    for sample in data:
        dataset_name = sample["dataset_name"]
        keys = keys_to_keep[dataset_name]  # which keys to keep for this sample's dataset

        filtered_sample = {key: sample[key] for key in keys}

        # apply renaming, if any is defined for this dataset
        for old_name, new_name in renaming_keys.get(dataset_name, []):
            filtered_sample[new_name] = filtered_sample.pop(old_name)

        filtered.append(filtered_sample)

    return filtered

keys_to_keep = {
    "stanford_hydra_dataset_converted_externally_to_rlds": ["actions", "image", "episode_idx", "dataset_name"],
    "taco_play": ["actions", "rgb_static", "episode_idx", "dataset_name"],
}

renaming_keys = {
    "taco_play": [("rgb_static", "image")],  # so both datasets share the key "image"
}

data_filtered = filter_keys(data_discretized, keys_to_keep, renaming_keys)

print(data_filtered[9760].keys())  # dict_keys(['actions', 'image', 'episode_idx', 'dataset_name'])
print(data_filtered[9761].keys())  # dict_keys(['actions', 'image', 'episode_idx', 'dataset_name'])

dict_keys(['actions', 'image', 'episode_idx', 'dataset_name'])
dict_keys(['actions', 'episode_idx', 'dataset_name', 'image'])


#### Resize images


In [8]:
def episode_to_ids(dataset: list[dict]):
  # key: (dataset_name, episode_idx); value: list of sample indices belonging to that episode
  episode_to_indices: dict[tuple[str, int], list[int]] = {}
  for i, sample in enumerate(dataset):
      key = (sample["dataset_name"], sample["episode_idx"])
      episode_to_indices.setdefault(key, []).append(i)
  return episode_to_indices

def plot_images(dataset: list[dict]):
    # Dict: keys=(data_name, episode_idx), value=[indices in dataset of pair (data_name, episode_idx)].
    episode_to_indices = episode_to_ids(dataset)

    for dataname_episode in episode_to_indices.keys():
        episode_1st_step = episode_to_indices[dataname_episode][0] # first step of each episode.
        data_name = dataset[episode_1st_step]['dataset_name']
        for image_key in metadata[dataname_episode[0]]['image_keys']:
            img = dataset[episode_1st_step][image_key]

            print(f"Dataset: {dataname_episode[0]}\nEpisode: {dataname_episode[1]}\nImage Key: {image_key}\nImage shape: {img.shape}\n\n")
            if img.shape[0] == 3:
                img = img.permute(1,2,0)
            
            plt.imshow(img)
            plt.show()

def image_resizing(data: list[dict], out_image_dim: int = 224):
    """
    Resize + normalize images in `data`, computing mean/std separately for
    each (dataset_name, episode_idx, image_key) group — since lighting/scene
    statistics can vary a lot across episodes and datasets, normalizing
    globally would blur that out.

    Modifies `data` in place: image arrays are replaced with normalized,
    resized torch.Tensor images of shape (3, out_image_dim, out_image_dim).
    """
    # group sample indices by (dataset_name, episode_idx)
    episode_to_indices: dict[tuple[str, int], list[int]] = {}
    for i, sample in enumerate(data):
        key = (sample["dataset_name"], sample["episode_idx"])
        episode_to_indices.setdefault(key, []).append(i)

    for (dataset_name, episode_idx), indices in episode_to_indices.items():
        # gather every image for this image_key within this one episode
        raw_images = [data[i]['image'] for i in indices]  # each (H, W, 3) uint8

        # convert to tensor stack (N, 3, H, W), scaled to [0, 1]
        episode_stack = torch.stack([TF.to_tensor(img) for img in raw_images])

        # print(episode_stack.shape)
        # per-channel mean/std across the whole episode (dims: N, H, W)
        mean = episode_stack.mean(dim=[0, 2, 3])
        std = episode_stack.std(dim=[0, 2, 3]).clamp(min=1e-6)  # avoid div-by-zero

        transform = transforms.Compose([
            transforms.Resize((out_image_dim, out_image_dim),
                                interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=mean, std=std),
        ])

        # apply resize+normalize to each image, write back into data in place
        for i, img_tensor in zip(indices, episode_stack):
            data[i]['image'] = transform(img_tensor)

    return data

data_filtered = image_resizing(data_filtered)


#### Dataloaders 
* split_by_episode(): Episodes appearing in training set must not appear in validation set.

In [9]:
def split_by_episode(dataset: list[dict], val_frac: float = 0.1, seed: int = 42):
    '''
    - For each sample, get its (dataset_name, episode_idx) pair, since episode_idx
      alone is not globally unique (it resets to 0 for each dataset_name).
    - Split episodes into train/val sets, constrained to not overlap.
    - Use sample indices to build train/val Subsets, enforcing different episodes
      (within each dataset_name) across the two splits.
    '''

    episode_to_indices = episode_to_ids(dataset)
    episode_ids = sorted(episode_to_indices.keys())  # sorted list of (dataset_name, episode_idx) tuples

    rng = np.random.default_rng(seed)
    rng.shuffle(episode_ids)

    # select different episodes for train vs validation
    n_val_ep = max(1, int(val_frac * len(episode_ids)))
    train_episodes = set(episode_ids[n_val_ep:]) # keys of episode_to_indices going to training data.
    val_episodes = set(episode_ids[:n_val_ep])
    

    # sample indices, i.e., indices of dataset[idx]
    train_indices = [i for ep in train_episodes for i in episode_to_indices[ep]]
    val_indices   = [i for ep in val_episodes   for i in episode_to_indices[ep]]

    return Subset(dataset, train_indices), Subset(dataset, val_indices)

def make_weights(subset, metadata):
  '''
  List of weights with length equal to subset length.
  Individual datasets stored within subset are not sorted.
  Loop over subset and save the weight associated with the
  corresponding dataset.  
  '''

  scaling_factor = 150 # to make weights not too small.

  n_samples_hydra = metadata['stanford_hydra_dataset_converted_externally_to_rlds']["total_samples"]
  n_taco_play = metadata['taco_play']["total_samples"]

  w_hydra = 1/n_samples_hydra * scaling_factor
  w_taco  = 1/n_taco_play * scaling_factor

  # per-dataset weight, indexed by dataset_name
  dataset_weight = {
      "stanford_hydra_dataset_converted_externally_to_rlds": w_hydra,
      "taco_play": w_taco,
  }
  
  return [dataset_weight[subset.dataset[i]["dataset_name"]] for i in subset.indices]

# get training and validation subsets. Force episodes to appear in train or val, but not both.
train_set, val_set = split_by_episode(data_filtered, val_frac=0.1, seed=42)

train_weights = make_weights(train_set, metadata)
val_weights   = make_weights(val_set, metadata)

train_sampler = WeightedRandomSampler(train_weights, num_samples=len(train_weights), replacement=True)
train_loader  = DataLoader(train_set, sampler=train_sampler, batch_size=32)

# no weighted sampling needed for validation. Just shuffle=False, full pass
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

'''
Note data is shuffled. Each sample within data list represents a time step with a pair (image, action).
However, within a single episode two consecutive pairs carry information as they are highly correlated
(i.e., the movement of the robotic arm is continuous).
TODO: Position encoding helps with this.
'''

'\nNote data is shuffled. Each sample within data list represents a time step with a pair (image, action).\nHowever, within a single episode two consecutive pairs carry information as they are highly correlated\n(i.e., the movement of the robotic arm is continuous).\nTODO: Position encoding helps with this.\n'

# Model

* Input: Image
    * OxE datasets contain different image resolutions -> Images must be same resolution.
    * Use for both datasets the 3rd-person view only (image for Hydra, rgb_static for taco_play).
    * Dimensions ([height, width, RGB]):
        * Hydra -> (240,320,3)
        * taco_play (rgb_static) ->  (150, 200, 3)
    * Normalize images:
        * Feeding your conv2d patchify layer inputs that are all positive means the gradients on that layer's weights tend to move in a more correlated biased direction (since the input's mean isn't 0), which slows down and destabilizes early training.
        * Consistency across datasets:
            * Images might differ in brightness, contrast or camera/sensor characteristics. Normalization puts them in equal footing.
    * 
* Output: Action token

* 7 DoF of action space are discretized in 256 bins.

### Archictecture:
* Image encoder (follow Dosovitskiy et al 2021, arXiv ~ ViT paper):
    * Overall architecture: CNN → extract patches from CNN feature map → linear projection to (D) → +pos embed + [class] token → Transformer encoder → head
    * PatchEmbedding():
        * CNN ~ Patchify:
        * Transform 2D images into sequence of patches (suitable for Transformers which have no built-in notion of spatial structure).
    * TransformerEncoderBlock():
        * MSA(LN(x)) + residual:
            * First apply <b>LayerNorm</b> to the current sequence representation (x).
                * 2016 - Layer Normalization - Lei Ba, Kiros, Hinton (https://arxiv.org/pdf/1607.06450)
            * Then apply <b>multi-head self-attention (MSA)</b>
            * Then add back the original (x) via a skip/<b>residual</b> connection.
    * MLP(LN(x)) + residual: this is the feed-forward (MLP) sublayer that follows attention within each Transformer encoder layer, again in pre-normalization form. You apply LayerNorm, then the MLP block, then add a residual connection (skip connection) from the block input. The paper also specifies that this MLP has two layers with a GELU non-linearity.
    * MLP head at the end of the transformer to project image embedding onto language model input space (256 tokens).
    * Notes:
        * Self-attention itself is permutation-invariant — it has no idea that patch 5 is next to patch 6, or that patch 1 is top-left and patch 196 is bottom-right. Without positional information, shuffling all the patches would produce the identical output. So after patchifying, you add a learned (or fixed sinusoidal) positional embedding per patch position

* ActionPrediction:
    * Use pre-trained language model.

* <b>TODO</b>:
    * Add layer tracking the name of the dataset.

In [10]:
# class LanguageEncoder(nn.Module):
#     '''
#     TODO: Encode Language Instructions. Incorporate it in the second model.
#     '''
#     def __init__(self):
#         ...


# Output dimension must be input dimension of language model (256 to match with OpenVLA model).

class PatchEmbedding(nn.Module):
    """
    Splits an image into fixed-size patches and linearly projects each to
    dimension D — Eq. 1 in the paper: x_p in R^(N x (P^2*C)) -> R^(N x D).

    Implemented as a single strided Conv2d, which is mathematically identical
    to "flatten each PxP patch, then apply a shared linear layer" but avoids
    the explicit flatten/unfold step: a Conv2d with kernel_size=stride=P applied
    to a non-overlapping patch produces exactly one linear projection per patch.
    """

    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_channels: int = 3, embed_dim: int = 768):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"

        self.num_patches = (img_size // patch_size) ** 2  # N = HW / P^2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                               kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W) -> (B, D, H/P, W/P) -> (B, N, D)
        x = self.proj(x)                     # (B, D, H/P, W/P)
        x = x.flatten(2)                     # (B, D, N)
        x = x.transpose(1, 2)                # (B, N, D)  -- patch embeddings, sequence-first
        return x


class TransformerEncoderBlock(nn.Module):
    """
    One Transformer encoder layer, pre-norm as specified in (Dosovitsky et al 2021, arXiv):
        z'_l = MSA(LN(z_{l-1})) + z_{l-1}      (Eq. 2)
        z_l  = MLP(LN(z'_l)) + z'_l            (Eq. 3)
    MLP is two linear layers with a GELU nonlinearity in between.
    """

    def __init__(self, embed_dim: int, num_heads: int, mlp_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # --- MSA(LN(x)) + residual ---
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed, need_weights=False)
        x = x + attn_out

        # --- MLP(LN(x)) + residual ---
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """
    Full ViT encoder: patchify -> [class] token + positional embedding ->
    L transformer encoder blocks -> final LayerNorm -> image representation y.

    Follows Eq. 1-4:
        z0   = [x_class; x_p^1 E; ...; x_p^N E] + E_pos
        z'_l = MSA(LN(z_{l-1})) + z_{l-1},  l = 1..L
        z_l  = MLP(LN(z'_l)) + z'_l,        l = 1..L
        y    = LN(z_L^0)                     <- state of [class] token, final layer
    
    References:
        AN IMAGE IS WORTH 16X16 WORDS: TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE - Dosovitskiy
    """

    def __init__(
        self,
        img_size: int = 224, 
        patch_size: int = 16,
        in_channels: int = 3,
        embed_dim: int = 768, # Table 1 in Dosovitsky et al 2021, arXiv. Hidden size D, output of CNN.
        num_layers: int = 12, # Table 1 in Dosovitsky et al 2021, arXiv.
        num_heads: int = 12, # Table 1 in Dosovitsky et al 2021, arXiv.
        mlp_dim: int = 3072,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        # learnable [class] token, prepended to the patch sequence (Section 3.1)
        self.class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # standard learnable 1D position embeddings (paper found 2D-aware embeddings
        # gave no significant improvement, Appendix D.4) -- one per position,
        # including the [class] token position (hence num_patches + 1)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        self.dropout = nn.Dropout(dropout)

        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)  # Eq. 4: y = LN(z_L^0)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.class_token, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 3, H, W) -- a batch of preprocessed images.
        Returns: (B, D) -- the [class] token's final representation, y.
        """
        B = x.shape[0]

        x = self.patch_embed(x)                              # (B, N, D)

        class_tokens = self.class_token.expand(B, -1, -1)     # (B, 1, D)
        x = torch.cat([class_tokens, x], dim=1)                # (B, N+1, D)

        x = x + self.pos_embed                                 # add positional info
        x = self.dropout(x)

        for block in self.encoder_blocks:
            x = block(x)

        x = self.final_norm(x)                                  # Eq. 4

        return x[:, 0]  # y: representation of the [class] token, shape (B, D)


In [ ]:
class ActionHead(nn.Module):
    """
    Small MLP head mapping the ViT's image representation (y) to an action
    vector. Given actions are already discretized to [0, 255] bins per
    dimension (from earlier preprocessing), this predicts per-dimension
    logits over 256 classes -- i.e. classification, not regression.
    """

    def __init__(self, embed_dim: int, action_dim: int, n_bins: int = 256, hidden_dim: int = 512):
        super().__init__()
        self.action_dim = action_dim
        self.n_bins = n_bins

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, action_dim * n_bins), # the output dimension matches that of the action space.
        )

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        # y: (B, D) -> (B, action_dim, n_bins) logits, one softmax per action dim
        B = y.shape[0]
        out = self.mlp(y)
        return out.view(B, self.action_dim, self.n_bins) # for a given action dimension, the probability of having a specific bin (from 0 - 255).

In [ ]:
class RoboticActionModel(nn.Module):
    """
    ViT image encoder + action head. Given your two datasets have different
    action_dim, instantiate one ActionHead per dataset and route by dataset_name.
    """

    def __init__(self, vit: VisionTransformer, action_heads: dict[str, ActionHead]):
        super().__init__()
        self.vit = vit
        self.action_heads = nn.ModuleDict(action_heads)  # keyed by dataset_name

    def forward(self, images: torch.Tensor, dataset_name: str) -> torch.Tensor:
        y = self.vit(images)                          # (B, D) shared image encoder
        return self.action_heads[dataset_name](y)      # (B, action_dim, n_bins)


# Example instantiation, ViT-Base config from Table 1 (12 layers, D=768, MLP=3072, 12 heads)
vit = VisionTransformer(
    img_size=224, patch_size=16, in_channels=3,
    embed_dim=768, num_layers=12, num_heads=12, mlp_dim=3072, dropout=0.1,
)

action_heads = {
    "stanford_hydra_dataset_converted_externally_to_rlds": ActionHead(embed_dim=768, action_dim=7),   # from metadata["...hydra..."]["action_keys"]
    "taco_play": ActionHead(embed_dim=768, action_dim=7),  # adjust action_dim per that dataset's actual action vector size
}

model = RoboticActionModel(vit, action_heads)

# forward pass
batch = next(iter(train_loader))
logits = model(batch["image"], dataset_name=batch["dataset_name"][0])  # (B, action_dim, 256)

# Training

* TODO:
    - Which optimizer?
    - Which loss function?

#### Loss function

In [ ]:
# ---------------------------------------------------------------------------
# Loss: cross-entropy over discretized action tokens only
# ---------------------------------------------------------------------------
def action_ce_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    logits:  (B, action_dim, n_bins) -- per-dimension bin logits from ActionHead
    targets: (B, action_dim)          -- discretized bin indices (int64), in [0, n_bins-1]

    nn.functional.cross_entropy expects (B, C, d1, ..., dk) logits and
    (B, d1, ..., dk) integer targets when doing multi-dim classification,
    with C (n_bins here) as dim=1. Our logits have n_bins as the LAST dim,
    so we permute (B, action_dim, n_bins) -> (B, n_bins, action_dim) first.
    """
    logits = logits.permute(0, 2, 1)  # (B, n_bins, action_dim)
    return F.cross_entropy(logits, targets)  # mean over batch AND action_dim


def action_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    """Per-token (per action-dim, per sample) top-1 accuracy, averaged."""
    preds = logits.argmax(dim=-1)  # (B, action_dim)
    return (preds == targets).float().mean().item()

In [ ]:
# DataLoader

# Evaluation 

# Tests (Supplementary).

#### Resizing Image 

In [59]:
np.random.seed(3)

# define array and convert to tensor
arr = np.random.randint(low=0, high=14, size=(200,3,234,300)).astype(np.float64)
tensor = torch.from_numpy(arr)

out_image_dim = 10
transform = transforms.Resize((out_image_dim, out_image_dim), interpolation=transforms.InterpolationMode.BILINEAR)

print(tensor.shape)
print(transform(tensor).shape)

torch.Size([200, 3, 234, 300])
torch.Size([200, 3, 10, 10])


#### Normalization Image

In [ ]:
np.random.seed(3)

# define array and convert to tensor
arr = np.random.randint(low=0, high=14, size=(3,4,4)).astype(np.float64)
tensor = torch.from_numpy(arr)

# built normalization
transform = transforms.Normalize(mean=tensor.mean(dim=[1,2]),std=tensor.std(dim=[1,2]))

print(tensor)
print(tensor.mean(dim=[1,2]), tensor.std(dim=[1,2]))

print(transform(tensor))
print(transform(tensor).mean(dim=[1,2]), transform(tensor).std(dim=[1,2]))

tensor([[[10.,  8.,  9.,  3.],
         [ 8.,  8.,  0.,  5.],
         [13.,  3., 10., 11.],
         [ 9.,  9., 10.,  5.]],

        [[ 7.,  6.,  0.,  4.],
         [12., 13.,  7.,  8.],
         [10.,  1., 10.,  6.],
         [ 2.,  2., 12.,  1.]],

        [[ 3., 10.,  5.,  8.],
         [ 1., 13., 10.,  8.],
         [11.,  7.,  8., 11.],
         [ 1., 11.,  0.,  5.]]], dtype=torch.float64)
tensor([7.5625, 6.3125, 7.0000], dtype=torch.float64) tensor([3.4442, 4.3162, 4.0825], dtype=torch.float64)
tensor([[[ 0.7077,  0.1270,  0.4174, -1.3247],
         [ 0.1270,  0.1270, -2.1957, -0.7440],
         [ 1.5787, -1.3247,  0.7077,  0.9981],
         [ 0.4174,  0.4174,  0.7077, -0.7440]],

        [[ 0.1593, -0.0724, -1.4625, -0.5358],
         [ 1.3177,  1.5494,  0.1593,  0.3910],
         [ 0.8543, -1.2308,  0.8543, -0.0724],
         [-0.9992, -0.9992,  1.3177, -1.2308]],

        [[-0.9798,  0.7348, -0.4899,  0.2449],
         [-1.4697,  1.4697,  0.7348,  0.2449],
         [ 0.9798, 